# 04. Generalized Linear Mixed Models (GLMM) & Positional Dominance

## Econometric / Statistical Specification
To account for repeated-measures dependency (keywords nested within topics and participants), we estimate a Generalized Linear Mixed Model with Binomial Logit link:

$$	ext{logit}(P(Y_{ijk} = 1)) = eta_0 + eta_{	ext{pos}} \cdot 	ext{Position}_{ijk} + eta_{	ext{weight}} \cdot 	ext{Weight}_{ijk} + \mathbf{X} oldsymbol{\gamma} + u_i + v_j + arepsilon_{ijk}$$

where $u_i \sim \mathcal{N}(0, \sigma_u^2)$ is the random participant intercept and $v_j \sim \mathcal{N}(0, \sigma_v^2)$ is the random topic intercept.


In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

df = pd.read_csv('/mnt/data/h2_testing_dataset.csv')


In [2]:
# 1. Standard Logistic Regression with Clustered Standard Errors
logit_mod = smf.logit("selected ~ displayed_position + model_weight + C(condition)", data=df).fit(cov_type='cluster', cov_kwds={'groups': df['participant_id']})
print(logit_mod.summary())


Optimization terminated successfully.
         Current function value: 0.243996
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:               selected   No. Observations:                18000
Model:                          Logit   Df Residuals:                    17995
Method:                           MLE   Df Model:                            4
Date:                Fri, 02 Oct 2026   Pseudo R-squ.:                  0.4228
Time:                        17:48:05   Log-Likelihood:                -4391.9
converged:                       True   LL-Null:                       -7608.8
Covariance Type:              cluster   LLR p-value:                     0.000
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                    1.4842      0.152      9.769      0.000       1.186

In [3]:
# 2. Linear Mixed Model (MixedLM) for Decomposition
mixed_mod = smf.mixedlm("selected ~ displayed_position + model_weight + C(condition)", 
                        data=df, 
                        groups=df["participant_id"]).fit()
print(mixed_mod.summary())


/usr/local/lib/python3.11/site-packages/statsmodels/base/model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
/usr/local/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/usr/local/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2054: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/usr/local/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/usr/local/lib/python3.11/site-packages/statsmodels/r

                     Mixed Linear Model Regression Results
Model:                     MixedLM         Dependent Variable:         selected
No. Observations:          18000           Method:                     REML    
No. Groups:                60              Scale:                      0.0933  
Min. group size:           300             Log-Likelihood:             inf     
Max. group size:           300             Converged:                  Yes     
Mean group size:           300.0                                               
-------------------------------------------------------------------------------
                         Coef.   Std.Err.     z    P>|z|    [0.025     0.975]  
-------------------------------------------------------------------------------
Intercept                 0.000  96624.939   0.000 1.000 -189381.400 189381.400
C(condition)[T.Reverse]  -0.000 165199.738  -0.000 1.000 -323785.536 323785.536
C(condition)[T.Shuffled]  0.000 141229.955   0.000 1.000 -276

In [4]:
# 3. Relative Contribution Ratio (Standardized Beta Analysis)
std_df = df.copy()
std_df['displayed_position_std'] = (df['displayed_position'] - df['displayed_position'].mean()) / df['displayed_position'].std()
std_df['model_weight_std'] = (df['model_weight'] - df['model_weight'].mean()) / df['model_weight'].std()

std_logit = smf.logit("selected ~ displayed_position_std + model_weight_std", data=std_df).fit()
print(" --- STANDARDIZED EFFECT SIZES ---")
print(std_logit.summary())
beta_pos = std_logit.params['displayed_position_std']
beta_weight = std_logit.params['model_weight_std']
print(f" Standardized Beta Ratio (|Pos| / Weight): {abs(beta_pos) / abs(beta_weight):.3f}")


Optimization terminated successfully.
         Current function value: 0.243996
         Iterations 9
 --- STANDARDIZED EFFECT SIZES ---
                           Logit Regression Results                           
Dep. Variable:               selected   No. Observations:                18000
Model:                          Logit   Df Residuals:                    17997
Method:                           MLE   Df Model:                            2
Date:                Fri, 02 Oct 2026   Pseudo R-squ.:                  0.4228
Time:                        17:48:08   Log-Likelihood:                -4391.9
converged:                       True   LL-Null:                       -7608.8
Covariance Type:            nonrobust   LLR p-value:                     0.000
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -3.7480      0.067    -